# IEEE-30 Dispatch Optimization Validation

This notebook demonstrates the predict-then-optimize pipeline for power grid dispatch optimization using pretrained GridFM models.

**Objective**: Minimize redispatch cost and line overloads

$
\min_u \, \|u-u_{\text{base}}\|_2^2 + \lambda \rho_{\text{overload}}(\hat{v})
$

where:
- $u$ = active generation on PV buses (decision variable)
- $\hat{v} = \Phi_\theta(u)$ = predicted bus state from neural solver
- $\rho_{\text{overload}}$ = total line overload penalty

## Setup

In [18]:
import sys
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Add repo to path - work from notebook directory
notebook_dir = Path('.').resolve()
# Navigate up to repo root (from experiments/test/ -> repo root)
# experiments/test -> experiments -> repo root (GridFM-graphkit)
repo_root = notebook_dir.parent.parent
sys.path.insert(0, str(repo_root))

# Verify we can import from repo
print(f"Notebook dir: {notebook_dir}")
print(f"Repo root: {repo_root}")
print(f"Python path: {sys.path[0]}")

# Import pipeline modules
from experiments.test import (
    ScenarioData,
    extract_scenario_from_batch,
    PVDispatchDecisionSpec,
    NeuralSolverWrapper,
    OverloadPenaltyEvaluator,
    DispatchOptimizationProblem,
    PipelineValidationHarness,
)

from gridfm_graphkit.io.param_handler import NestedNamespace, load_normalizer, load_model
from gridfm_graphkit.datasets.powergrid_datamodule import LitGridDataModule

import yaml

print("✓ Imports successful")

Notebook dir: C:\Users\Caleb Lu\OneDrive\Documents\GT\Extracurriculars\Research\Grid FM\Experiments\GridFM-graphkit\experiments\test
Repo root: C:\Users\Caleb Lu\OneDrive\Documents\GT\Extracurriculars\Research\Grid FM\Experiments\GridFM-graphkit
Python path: C:\Users\Caleb Lu\OneDrive\Documents\GT\Extracurriculars\Research\Grid FM\Experiments\GridFM-graphkit
✓ Imports successful


## 1. Load Test Data (IEEE-30)

In [19]:
# Load configuration
config_path = repo_root / "tests" / "config" / "gridFMv0.1_dummy.yaml"

with open(config_path, 'r') as f:
    config_dict = yaml.safe_load(f)

args = NestedNamespace(**config_dict)

# Load datamodule - use test data directory
data_dir = repo_root / "tests" / "data"
datamodule = LitGridDataModule(args, data_dir=str(data_dir))
datamodule.setup(stage="test")

# Get a test batch
test_loader = datamodule.test_dataloader()[0]  # First dataset
batch = next(iter(test_loader))

print(f"✓ Loaded test batch")
print(f"  Batch keys: {batch.keys}")
print(f"  Num nodes: {batch.num_nodes}")
print(f"  Num edges: {batch.num_edges}")
print(f"  PE shape: {batch.pe.shape}")

✓ Loaded test batch
  Batch keys: <bound method BaseData.keys of DataBatch(x=[60, 9], edge_index=[2, 222], edge_attr=[222, 2], y=[60, 6], scenario_id=[2], edge_weight=[222], pe=[60, 20], mask=[60, 6], batch=[60], ptr=[3])>
  Num nodes: 60
  Num edges: 222
  PE shape: torch.Size([60, 20])


## 2. Extract Canonical Scenario

In [20]:
# Extract scenario from batch
node_normalizer = datamodule.node_normalizers[0]
edge_normalizer = datamodule.edge_normalizers[0]

scenario = extract_scenario_from_batch(
    batch,
    node_normalizer,
    edge_normalizer,
    scenario_idx=0,
    scenario_id="IEEE-30-test",
)

print(f"✓ Extracted scenario: {scenario.scenario_id}")
print(f"  Num buses: {scenario.num_buses}")
print(f"  PQ buses: {np.sum(scenario.PQ_mask)}")
print(f"  PV buses: {np.sum(scenario.PV_mask)}")
print(f"  REF buses: {np.sum(scenario.REF_mask)}")
print(f"  PE dimension: {scenario.pe.shape[1]}")

✓ Extracted scenario: IEEE-30-test
  Num buses: 60
  PQ buses: 48
  PV buses: 10
  REF buses: 2
  PE dimension: 20


## 3. Decision Variable Specification

In [21]:
# Define decision variables
decision_spec = PVDispatchDecisionSpec(scenario)

print(f"✓ Decision specification created")
print(f"  Number of PV buses: {decision_spec.n_pv}")
print(f"  PV bus indices: {decision_spec.pv_bus_indices}")
print(f"  Baseline Pg at PV buses: {decision_spec.u_base}")
print(f"  Min bounds: {decision_spec.u_min}")
print(f"  Max bounds: {decision_spec.u_max}")

summary = decision_spec.get_summary()
print(f"\nDecision spec summary:")
for key, val in summary.items():
    print(f"  {key}: {val:.4f}")

✓ Decision specification created
  Number of PV buses: 10
  PV bus indices: [ 1  4  7 10 12 31 34 37 40 42]
  Baseline Pg at PV buses: [1.0029768e-06 3.7129796e-10 6.4983052e-10 1.4867026e-09 8.6981072e-10
 8.5840549e-07 5.0265753e-11 3.3167236e-10 1.1350252e-09 3.9274464e-10]
  Min bounds: [8.0238141e-07 2.9703837e-10 5.1986443e-10 1.1893622e-09 6.9584860e-10
 6.8672438e-07 4.0212604e-11 2.6533789e-10 9.0802016e-10 3.1419572e-10]
  Max bounds: [1.2035722e-06 4.4555756e-10 7.7979667e-10 1.7840432e-09 1.0437730e-09
 1.0300867e-06 6.0318910e-11 3.9800685e-10 1.3620303e-09 4.7129362e-10]

Decision spec summary:
  n_pv: 10.0000
  u_base_mean: 0.0000
  u_base_min: 0.0000
  u_base_max: 0.0000
  u_min_mean: 0.0000
  u_max_mean: 0.0000
  bound_range_mean: 0.0000


## 4. Load Neural Solver Models

In [22]:
# Load pretrained models
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# GNN model (v0.1)
model_gnn = load_model(args)
model_gnn_path = repo_root / "examples" / "models" / "GridFM_v0_1.pth"

if model_gnn_path.exists():
    checkpoint = torch.load(model_gnn_path, map_location=device)
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        model_gnn.load_state_dict(checkpoint['state_dict'], strict=False)
    else:
        model_gnn.load_state_dict(checkpoint, strict=False)
    print(f"✓ Loaded GNN model from {model_gnn_path}")
else:
    print(f"⚠ GNN model not found at {model_gnn_path}")

# Create neural solver wrapper for GNN
solver_gnn = NeuralSolverWrapper(
    model_gnn,
    model_type="gnn",
    scenario=scenario,
    decision_spec=decision_spec,
    device=device,
)

print(f"✓ Created GNN solver wrapper")

Device: cpu
✓ Loaded GNN model from C:\Users\Caleb Lu\OneDrive\Documents\GT\Extracurriculars\Research\Grid FM\Experiments\GridFM-graphkit\examples\models\GridFM_v0_1.pth
✓ Created GNN solver wrapper


C:\Users\Caleb Lu\AppData\Local\Temp\ipykernel_6280\2498763988.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_gnn_path, map_location=devi

## 5. Create Overload Penalty Evaluator

In [23]:
# Create overload evaluator
overload_eval = OverloadPenaltyEvaluator(scenario, sn_mva=100.0)

print(f"✓ Created overload evaluator")

# Evaluate baseline overload
baseline_overload = overload_eval.evaluate_baseline()
print(f"\nBaseline overload metrics:")
for key, val in baseline_overload.items():
    print(f"  {key}: {val}")

✓ Created overload evaluator

Baseline overload metrics:
  total_penalty: 34181.484247517816
  n_overloaded_lines: 130
  max_loading: 53.29280901382795
  max_overload: 52.29280901382795
  mean_loading: 6.374274237128098


## 6. Full Pipeline Validation

In [24]:
# Run full validation
validation_report = PipelineValidationHarness.full_validation(
    scenario,
    decision_spec,
    solver_gnn,
    solver_gps=None,
    overload_eval=overload_eval,
)

# Print report
PipelineValidationHarness.print_validation_report(validation_report, verbose=True)

PIPELINE VALIDATION REPORT

[SCENARIO]
  Status: ✓ PASS (14/14)
    ✓ num_buses_consistent
    ✓ Pd_shape
    ✓ Qd_shape
    ✓ Pg_shape
    ✓ Qg_shape
    ✓ Vm_shape
    ✓ Va_shape
    ✓ PQ_shape
    ✓ PV_shape
    ✓ REF_shape
    ✓ G_shape
    ✓ B_shape
    ✓ pe_pe_dim
    ✓ mask_shape
    bus_types_mutually_exclusive: True
    has_ref_bus: True
    Pg_bounds_tight: True
    Pg_bounds_tight_2: True

[DECISION_SPEC]
  Status: ✓ PASS (4/4)
    ✓ n_pv_positive
    ✓ u_base_shape
    ✓ u_min_shape
    ✓ u_max_shape
    bounds_consistent: True
    u_base_within_bounds: True

[SOLVER_GNN]
  Status: ✓ PASS (12/12)
    ✓ model_type_valid
    ✓ model_in_eval
    ✓ prediction_successful
    ✓ prediction_keys
    ✓ pred_Pd_shape
    ✓ pred_Qd_shape
    ✓ pred_Pg_shape
    ✓ pred_Qg_shape
    ✓ pred_Vm_shape
    ✓ pred_Va_shape
    ✓ predictions_finite
    ✓ baseline_validation_successful
    Vm_reasonable_range: False
    baseline_errors: {'Pd_rmse': 131.77003479003906, 'Pd_mae': 75.310791015625

## 7. Test Solver Predictions

In [25]:
# Test prediction on baseline
u_base = decision_spec.u_base
pred = solver_gnn.predict_state(u_base)

print("Baseline prediction (first 5 buses):")
print(f"  Pd: {pred['Pd'][:5]}")
print(f"  Qd: {pred['Qd'][:5]}")
print(f"  Pg: {pred['Pg'][:5]}")
print(f"  Qg: {pred['Qg'][:5]}")
print(f"  Vm: {pred['Vm'][:5]}")
print(f"  Va: {pred['Va'][:5]}")

# Compute error vs baseline
baseline_state = scenario.get_baseline_state()
print(f"\nPrediction errors (RMSE):")
for key in ['Pd', 'Qd', 'Pg', 'Qg', 'Vm', 'Va']:
    rmse = np.sqrt(np.mean((pred[key] - baseline_state[key]) ** 2))
    print(f"  {key}: {rmse:.4f}")

Baseline prediction (first 5 buses):
  Pd: [-43.477962   -7.4643064 236.81299   -76.02536    49.86081  ]
  Qd: [-182.97997 -469.03284 -515.6699  -389.3066  -160.416  ]
  Pg: [-45.249405 182.30252  287.1426   113.54898   10.183813]
  Qg: [ 51.49936    70.222725  345.22      142.87143     4.6817164]
  Vm: [ 1.1932938   0.16370894 -0.3523711   0.33155808  0.02016059]
  Va: [ -29.901863  -30.769012   52.55691  -165.72824    -5.607497]

Prediction errors (RMSE):
  Pd: 131.7700
  Qd: 341.2513
  Pg: 149.5372
  Qg: 148.4070
  Vm: 0.9450
  Va: 33.6066


## 8. Create Optimization Problem

In [26]:
# Create optimization problem
problem = DispatchOptimizationProblem(
    scenario=scenario,
    decision_spec=decision_spec,
    solver=solver_gnn,
    overload_eval=overload_eval,
    alpha=0.6,      # Weight on baseline deviation
    lambda_=0.4,    # Weight on overload penalty
)

print(f"✓ Created optimization problem")
print(f"  alpha (baseline deviation weight): {problem.alpha}")
print(f"  lambda (overload penalty weight): {problem.lambda_}")

✓ Created optimization problem
  alpha (baseline deviation weight): 0.6
  lambda (overload penalty weight): 0.4


## 9. Evaluate Objective at Baseline

In [27]:
# Evaluate objective at baseline
u_base = decision_spec.u_base
obj_base, details_base = problem.objective(u_base, return_details=True)

print(f"Objective at baseline dispatch:")
print(f"  Objective value: {obj_base:.4f}")
print(f"  Baseline deviation cost: {details_base['cost_deviation']:.4f}")
print(f"  Overload penalty: {details_base['penalty_overload']:.4f}")
print(f"  Number of overloaded lines: {details_base['n_overloaded_lines']}")
print(f"  Max line loading: {details_base['max_loading']:.4f}")

Objective at baseline dispatch:
  Objective value: 66526.5599
  Baseline deviation cost: 0.0000
  Overload penalty: 166316.3997
  Number of overloaded lines: 132
  Max line loading: 156.9162


## 10. Run Optimization

In [28]:
# Run optimization (with reduced iterations for demo)
print("Starting optimization...")
opt_result = problem.optimize(
    method="L-BFGS-B",
    maxiter=50,
    verbose=True,
)

print(f"\nOptimization result:")
print(f"  Success: {opt_result['success']}")
print(f"  Message: {opt_result['message']}")
print(f"  Iterations: {opt_result['n_iter']}")
print(f"  Final objective: {opt_result['obj_opt']:.4f}")
print(f"  Final cost: {opt_result['cost_opt']:.4f}")
print(f"  Final penalty: {opt_result['penalty_opt']:.4f}")

Starting optimization...

Optimization result:
  Success: True
  Message: CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
  Iterations: 0
  Final objective: 66526.5599
  Final cost: 0.0000
  Final penalty: 166316.3997


## 11. Compare Baseline vs. Optimized

In [29]:
# Compare baseline vs optimized
u_opt = opt_result['u_opt']
comparison = problem.compare_baseline_vs_optimized(u_opt)

print("\n" + "="*70)
print("BASELINE vs OPTIMIZED COMPARISON")
print("="*70)

print(f"\nBASELINE:")
for key, val in comparison['baseline'].items():
    if key != 'u':
        print(f"  {key}: {val:.4f}")

print(f"\nOPTIMIZED:")
for key, val in comparison['optimized'].items():
    if key != 'u':
        print(f"  {key}: {val:.4f}")

print(f"\nIMPROVEMENT:")
for key, val in comparison['improvement'].items():
    print(f"  {key}: {val:.4f}")

print("\n" + "="*70)


BASELINE vs OPTIMIZED COMPARISON

BASELINE:
  objective: 66526.5599
  cost: 0.0000
  penalty: 166316.3997
  n_overloaded: 132.0000
  max_loading: 156.9162

OPTIMIZED:
  objective: 66526.5599
  cost: 0.0000
  penalty: 166316.3997
  n_overloaded: 132.0000
  max_loading: 156.9162

IMPROVEMENT:
  objective: 0.0000
  objective_pct: 0.0000
  penalty: 0.0000
  penalty_pct: 0.0000



## 12. Visualize Optimization Progress

In [30]:
# Plot optimization history
history = opt_result['history']

if len(history['cost']) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Objective
    axes[0, 0].plot(history['objective'], 'b-', linewidth=2)
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Objective Value')
    axes[0, 0].set_title('Total Objective')
    axes[0, 0].grid(True)
    
    # Cost
    axes[0, 1].plot(history['cost'], 'g-', linewidth=2)
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].set_ylabel('Baseline Deviation Cost')
    axes[0, 1].set_title('Cost Component')
    axes[0, 1].grid(True)
    
    # Penalty
    axes[1, 0].plot(history['penalty'], 'r-', linewidth=2)
    axes[1, 0].set_xlabel('Iteration')
    axes[1, 0].set_ylabel('Overload Penalty')
    axes[1, 0].set_title('Penalty Component')
    axes[1, 0].grid(True)
    
    # Dispatch changes
    u_values = np.array(history['u'])
    for i in range(min(3, u_values.shape[1])):
        axes[1, 1].plot(u_values[:, i], label=f'PV bus {i}')
    axes[1, 1].set_xlabel('Iteration')
    axes[1, 1].set_ylabel('Pg (pu)')
    axes[1, 1].set_title('Active Generation Changes')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig('optimization_progress.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Optimization progress plot saved as 'optimization_progress.png'")
else:
    print("No optimization history to plot")

No optimization history to plot


## 13. Summary and Next Steps

In [31]:
print("\n" + "="*70)
print("PHASE 1 DEMONSTRATE COMPLETE - SUMMARY")
print("="*70)

print("\n✓ Successfully demonstrated predict-then-optimize pipeline:")
print("  1. Loaded IEEE-30 test scenario")
print("  2. Extracted canonical scenario representation")
print("  3. Defined PV-bus Pg decision variables")
print("  4. Loaded pretrained GNN model")
print("  5. Created neural solver wrapper")
print("  6. Implemented overload penalty evaluation")
print("  7. Validated pipeline components")
print("  8. Ran baseline-vs-optimized dispatch optimization")

print(f"\nKey results:")
print(f"  Baseline objective: {comparison['baseline']['objective']:.4f}")
print(f"  Optimized objective: {comparison['optimized']['objective']:.4f}")
print(f"  Improvement: {comparison['improvement']['objective_pct']:.2f}%")
print(f"  Overload reduction: {comparison['improvement']['penalty_pct']:.2f}%")

print("\n📋 Next steps:")
print("  • Load GPS_v0.2 model and compare solutions")
print("  • Test with larger scenarios (case57, case118, etc.)")
print("  • Investigate generator bounds sourcing")
print("  • Tune hyperparameters (alpha, lambda)")
print("  • Add constraint handling for full feasibility")
print("  • Extend to REF-bus participation (phase 2)")
print("  • Consider Vm setpoint optimization (phase 2)")
print("\n" + "="*70)


PHASE 1 DEMONSTRATE COMPLETE - SUMMARY

✓ Successfully demonstrated predict-then-optimize pipeline:
  1. Loaded IEEE-30 test scenario
  2. Extracted canonical scenario representation
  3. Defined PV-bus Pg decision variables
  4. Loaded pretrained GNN model
  5. Created neural solver wrapper
  6. Implemented overload penalty evaluation
  7. Validated pipeline components
  8. Ran baseline-vs-optimized dispatch optimization

Key results:
  Baseline objective: 66526.5599
  Optimized objective: 66526.5599
  Improvement: 0.00%
  Overload reduction: 0.00%

📋 Next steps:
  • Load GPS_v0.2 model and compare solutions
  • Test with larger scenarios (case57, case118, etc.)
  • Investigate generator bounds sourcing
  • Tune hyperparameters (alpha, lambda)
  • Add constraint handling for full feasibility
  • Extend to REF-bus participation (phase 2)
  • Consider Vm setpoint optimization (phase 2)

